# Ablation Results Visualization (No Optuna)

This notebook loads all ablation runs from:

`/scratch/work/sethih1/ablation_runs_no_optuna/runs`

and generates publication-ready tables/figures for the research paper.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
# Adjust these only if your paths differ
RUNS_DIR = Path('/scratch/work/sethih1/ablation_runs_no_optuna/runs')
OUTPUT_DIR = Path('/scratch/work/sethih1/Multimodal_TERS/paper_results/ablation_no_optuna')
FIG_DIR = OUTPUT_DIR / 'figures'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('RUNS_DIR:', RUNS_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

## Load All Runs

In [ ]:
def safe_get(dct, path, default=np.nan):
    cur = dct
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return default
        cur = cur[p]
    return cur

run_rows = []
epoch_frames = []
missing = []

run_dirs = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()])

for run_dir in run_dirs:
    metrics_path = run_dir / 'metrics.json'
    epochs_path = run_dir / 'epoch_metrics.csv'

    if not metrics_path.exists() or not epochs_path.exists():
        missing.append(run_dir.name)
        continue

    with metrics_path.open('r', encoding='utf-8') as f:
        m = json.load(f)

    e = pd.read_csv(epochs_path)
    e['run_name'] = m.get('run_name', run_dir.name)
    e['ablation_name'] = m.get('ablation_name', np.nan)
    e['fusion_type'] = m.get('fusion_type', np.nan)
    e['freq_encoding'] = m.get('freq_encoding', np.nan)
    e['max_freqs'] = m.get('max_freqs', np.nan)
    epoch_frames.append(e)

    row = {
        'run_name': m.get('run_name', run_dir.name),
        'run_dir': str(run_dir),
        'ablation_name': m.get('ablation_name', np.nan),
        'fusion_type': m.get('fusion_type', np.nan),
        'freq_encoding': m.get('freq_encoding', np.nan),
        'max_freqs': m.get('max_freqs', np.nan),
        'seed': m.get('seed', np.nan),
        'best_epoch': m.get('best_epoch', np.nan),
        'best_val_macro_dice': m.get('best_val_macro_dice', np.nan),
        'val_loss': safe_get(m, ['val', 'loss']),
        'val_macro_dice': safe_get(m, ['val', 'macro_dice']),
        'val_min_class_dice': safe_get(m, ['val', 'min_class_dice']),
        'test_loss': safe_get(m, ['test', 'loss']),
        'test_macro_dice': safe_get(m, ['test', 'macro_dice']),
        'test_min_class_dice': safe_get(m, ['test', 'min_class_dice']),
        'epochs': m.get('epochs', np.nan),
        'batch_size': m.get('batch_size', np.nan),
        'lr': m.get('lr', np.nan),
        'loss_fn': m.get('loss_fn', np.nan),
        'num_channels': m.get('num_channels', np.nan),
        'threshold': m.get('threshold', np.nan),
        'model_params': m.get('model_params', np.nan),
        'device': m.get('device', np.nan),
    }

    val_pc = m.get('val', {}).get('per_class_dice', [])
    test_pc = m.get('test', {}).get('per_class_dice', [])

    class_cols = [c for c in e.columns if c.startswith('val_dice_')]
    class_names = [c.replace('val_dice_', '') for c in class_cols]

    for i, cls in enumerate(class_names):
        row[f'val_dice_{cls}'] = val_pc[i] if i < len(val_pc) else np.nan
        row[f'test_dice_{cls}'] = test_pc[i] if i < len(test_pc) else np.nan

    run_rows.append(row)

summary_df = pd.DataFrame(run_rows)
epoch_df = pd.concat(epoch_frames, ignore_index=True) if epoch_frames else pd.DataFrame()

ablation_label_map = {
    'none': 'Image-only Baseline',
    'early': 'Early Fusion',
    'late': 'Late Fusion',
    'attention': 'Attention Fusion',
    'film': 'FiLM Fusion',
    'hybrid': 'Hybrid Fusion',
    'freq_only': 'Frequency-only Branch',
}

config_label_map = {
    'binning-100': 'Frequency-Presence Vector',
}

max_freq_str = pd.to_numeric(summary_df['max_freqs'], errors='coerce').map(
    lambda x: str(int(x)) if pd.notna(x) else 'na'
)
summary_df['config_key'] = summary_df['freq_encoding'].astype(str) + '-' + max_freq_str
summary_df['config_label'] = summary_df['config_key'].map(config_label_map).fillna(summary_df['config_key'])
summary_df['ablation_label'] = summary_df['ablation_name'].map(ablation_label_map).fillna(summary_df['ablation_name'])

before_total = len(summary_df)
keep_max_freqs = {100}
summary_df = summary_df[pd.to_numeric(summary_df['max_freqs'], errors='coerce').isin(keep_max_freqs)].copy()
after_freq_filter = len(summary_df)

keep_config_keys = {'binning-100'}
summary_df = summary_df[summary_df['config_key'].isin(keep_config_keys)].copy()
after_repr_filter = len(summary_df)

exclude_ablations = {'freq_only'}
summary_df = summary_df[~summary_df['ablation_name'].isin(exclude_ablations)].copy()
after_ablation_filter = len(summary_df)

epoch_df = epoch_df[epoch_df['run_name'].isin(summary_df['run_name'])].copy()

print(f"Removed {before_total - after_freq_filter} run(s) due to max_freqs filter")
print(f"Removed {after_freq_filter - after_repr_filter} run(s) due to representation filter")
print(f"Removed {after_repr_filter - after_ablation_filter} run(s) due to ablation exclusion")
print(f"Kept max_freqs: {sorted(keep_max_freqs)}")
print(f"Kept representation: {sorted(keep_config_keys)}")
print(f"Excluded ablations: {sorted(exclude_ablations)}")
print(f'Loaded {len(summary_df)} runs after filtering')
if missing:
    print('Missing files in:', missing)

summary_df.head()

## Main Result Table

In [ ]:
display_cols = [
    'run_name', 'ablation_label', 'fusion_type',
    'test_macro_dice', 'test_min_class_dice', 'val_macro_dice',
    'best_epoch', 'model_params'
]

for c in ['test_dice_H', 'test_dice_C', 'test_dice_N', 'test_dice_O']:
    if c in summary_df.columns:
        display_cols.append(c)

main_table = summary_df[display_cols].sort_values(
    ['test_macro_dice', 'test_min_class_dice'], ascending=False
).reset_index(drop=True)

main_table = main_table.rename(columns={
    'ablation_label': 'ablation',
})

main_table.to_csv(OUTPUT_DIR / 'ablation_main_table.csv', index=False)
print('Wrote', OUTPUT_DIR / 'ablation_main_table.csv')

main_table

## Figure 1: Test Macro Dice by Ablation

In [ ]:
strategy_order = ['none', 'early', 'late', 'attention', 'film', 'hybrid']

ablation_label_lookup = (
    summary_df[['ablation_name', 'ablation_label']]
    .drop_duplicates()
    .set_index('ablation_name')['ablation_label']
    .to_dict()
)

plot_series = summary_df.groupby('ablation_name', as_index=True)['test_macro_dice'].mean()

fig, ax = plt.subplots(figsize=(10.5, 4.8))
x = np.arange(len(strategy_order))
vals = [plot_series.get(s, np.nan) for s in strategy_order]

bars = ax.bar(x, vals, color='#1f77b4', width=0.65)
for b, v in zip(bars, vals):
    if np.isfinite(v):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.006, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([ablation_label_lookup.get(s, s) for s in strategy_order], rotation=0, ha='center')
ax.set_ylabel('Test Macro Dice')
ax.grid(axis='y', alpha=0.25)
ax.set_ylim(0, min(1.0, np.nanmax(summary_df['test_macro_dice']) + 0.12))
plt.tight_layout()

out = FIG_DIR / 'fig1_test_macro_by_strategy_config.png'
fig.savefig(out, transparent=True)
print('Wrote', out)
plt.show()

## Figure 2: Macro vs Minimum-Class Tradeoff

In [ ]:
palette = {
    'none': '#4c4c4c',
    'early': '#d62728',
    'late': '#1f77b4',
    'attention': '#2ca02c',
    'film': '#ff7f0e',
    'hybrid': '#9467bd',
}

strategy_order = ['none', 'early', 'late', 'attention', 'film', 'hybrid']
marker_map = {'none': 'o', 'early': 'D', 'late': 'o', 'attention': '^', 'film': 's', 'hybrid': 'P'}
x_offset_map = {'late': -0.0012, 'film': 0.0012}

ablation_label_lookup = (
    summary_df[['ablation_name', 'ablation_label']]
    .drop_duplicates()
    .set_index('ablation_name')['ablation_label']
    .to_dict()
)

x = summary_df['test_macro_dice']
y = summary_df['test_min_class_dice']
xpad = max(0.01, 0.05 * (x.max() - x.min()))
ypad = max(0.01, 0.08 * (y.max() - y.min()))

fig, ax = plt.subplots(figsize=(7.2, 5.2))

for s in strategy_order:
    r = summary_df[summary_df['ablation_name'] == s]
    if r.empty:
        continue
    rr = r.iloc[0]
    x_plot = rr['test_macro_dice'] + x_offset_map.get(s, 0.0)
    y_plot = rr['test_min_class_dice']
    ax.scatter(
        x_plot,
        y_plot,
        marker=marker_map.get(s, 'o'),
        s=105,
        color=palette.get(s, '#333333'),
        edgecolor='black',
        linewidth=0.4,
        alpha=0.9,
        zorder=3,
    )

pts = summary_df[['test_macro_dice', 'test_min_class_dice']].dropna().sort_values('test_macro_dice')
pareto_x, pareto_y = [], []
best_y = -np.inf
for xx, yy in pts.values:
    if yy > best_y:
        pareto_x.append(xx)
        pareto_y.append(yy)
        best_y = yy
if len(pareto_x) >= 2:
    ax.plot(pareto_x, pareto_y, color='black', linestyle='--', linewidth=1.2, alpha=0.7, zorder=2)

if not summary_df.empty:
    best = summary_df.loc[summary_df['test_macro_dice'].idxmax()]
    ax.scatter(
        best['test_macro_dice'],
        best['test_min_class_dice'],
        marker='*',
        s=240,
        color='gold',
        edgecolor='black',
        linewidth=0.6,
        zorder=4,
    )

ax.set_xlabel('Test Macro Dice')
ax.set_ylabel('Test Minimum-Class Dice')
ax.grid(alpha=0.25)
ax.set_xlim(x.min() - xpad, x.max() + xpad)
ax.set_ylim(max(0.0, y.min() - ypad), min(1.0, y.max() + ypad))

legend_handles = [
    plt.Line2D([0], [0], marker=marker_map.get(s, 'o'), linestyle='None', markerfacecolor=palette[s], markeredgecolor='black', markeredgewidth=0.4, markersize=8, label=ablation_label_lookup.get(s, s))
    for s in strategy_order if s in summary_df['ablation_name'].values
]
legend_handles.append(
    plt.Line2D([0], [0], marker='*', linestyle='None', markerfacecolor='gold', markeredgecolor='black', markeredgewidth=0.6, markersize=10, label='Best macro point')
)
legend_handles.append(
    plt.Line2D([0], [0], linestyle='--', color='black', linewidth=1.2, label='Pareto front')
)

ax.legend(handles=legend_handles, loc='lower right', fontsize=8)
plt.tight_layout()

out = FIG_DIR / 'fig2_tradeoff_macro_vs_min.png'
fig.savefig(out, bbox_inches='tight', transparent=True)
print('Wrote', out)
plt.show()

## Figure 3: Per-Class Test Dice Matrix

In [ ]:
class_cols = [c for c in summary_df.columns if c.startswith('test_dice_')]
plot_df = summary_df.copy().sort_values('test_macro_dice', ascending=False)

if not class_cols:
    print('No per-class columns available in summary_df.')
else:
    mat = plot_df[class_cols].values
    row_labels = [r.ablation_label for r in plot_df.itertuples()]
    col_labels = [c.replace('test_dice_', '') for c in class_cols]

    fig, ax = plt.subplots(figsize=(8.5, max(4.2, 0.34 * len(row_labels))))
    im = ax.imshow(mat, aspect='auto', cmap='viridis', vmin=0, vmax=1)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=8)
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            if np.isfinite(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7, color='white' if v < 0.55 else 'black')

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Dice')

    plt.tight_layout()
    out = FIG_DIR / 'fig3_per_class_heatmap.png'
    fig.savefig(out, transparent=True)
    print('Wrote', out)
    plt.show()

## Figure 4: Validation Macro Dice Learning Curves

In [ ]:
top_runs = summary_df.sort_values('test_macro_dice', ascending=False).head(6)['run_name'].tolist()
curves = epoch_df[epoch_df['run_name'].isin(top_runs)].copy()

label_lookup = (
    summary_df[['run_name', 'ablation_label', 'config_label']]
    .drop_duplicates()
    .set_index('run_name')
    .to_dict(orient='index')
)

fig, ax = plt.subplots(figsize=(8.5, 5.5))

for run_name in top_runs:
    sub = curves[curves['run_name'] == run_name]
    if sub.empty:
        continue
    info = label_lookup.get(run_name, {})
    label = f"{info.get('ablation_label', run_name)} | {info.get('config_label', '')}"
    ax.plot(sub['epoch'], sub['val_macro_dice'], label=label, linewidth=1.7)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Macro Dice')
ax.grid(alpha=0.25)
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()

out = FIG_DIR / 'fig4_top_learning_curves.png'
fig.savefig(out, transparent=True)
print('Wrote', out)
plt.show()

## Figure 5: Gain vs Image-Only Baseline

In [ ]:
if summary_df.empty:
    print('No rows available after filtering.')
else:
    baseline = summary_df[summary_df['ablation_name'] == 'none']
    if baseline.empty:
        print('Image-only baseline (ablation_name=none) not found.')
    else:
        b = baseline.iloc[0]
        gain_rows = []
        for _, r in summary_df[summary_df['ablation_name'] != 'none'].iterrows():
            gain_rows.append({
                'ablation_name': r['ablation_name'],
                'ablation_label': r['ablation_label'],
                'macro_gain_vs_baseline': r['test_macro_dice'] - b['test_macro_dice'],
                'min_gain_vs_baseline': r['test_min_class_dice'] - b['test_min_class_dice'],
            })

        gain_df = pd.DataFrame(gain_rows).sort_values('macro_gain_vs_baseline', ascending=False).reset_index(drop=True)
        out_csv = OUTPUT_DIR / 'ablation_onehot_gain_vs_baseline.csv'
        gain_df.to_csv(out_csv, index=False)
        print('Wrote', out_csv)

        fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), sharey=False)
        xx = np.arange(len(gain_df))

        axes[0].bar(xx, gain_df['macro_gain_vs_baseline'], color='#1f77b4')
        axes[0].axhline(0, color='black', linewidth=1)
        axes[0].set_ylabel('Dice gain')
        axes[0].set_xticks(xx)
        axes[0].set_xticklabels(gain_df['ablation_label'], rotation=0, ha='center')
        axes[0].grid(axis='y', alpha=0.25)

        axes[1].bar(xx, gain_df['min_gain_vs_baseline'], color='#ff7f0e')
        axes[1].axhline(0, color='black', linewidth=1)
        axes[1].set_xticks(xx)
        axes[1].set_xticklabels(gain_df['ablation_label'], rotation=0, ha='center')
        axes[1].grid(axis='y', alpha=0.25)
        plt.tight_layout()

        out_fig = FIG_DIR / 'fig5_onehot_gain_vs_baseline.png'
        fig.savefig(out_fig, bbox_inches='tight', transparent=True)
        print('Wrote', out_fig)
        plt.show()

        gain_df

## Optional: Qualitative Panel (Top and Bottom Runs)

In [ ]:
def first_image_for_run(run_dir, split='test'):
    q = Path(run_dir) / 'qualitative'
    if not q.exists():
        return None
    cands = sorted(q.glob(f'{split}_sample_*.png'))
    if cands:
        return cands[0]
    cands = sorted(q.glob('*.png'))
    return cands[0] if cands else None

picked = pd.concat([
    summary_df.sort_values('test_macro_dice', ascending=False).head(3),
    summary_df.sort_values('test_macro_dice', ascending=True).head(3),
], ignore_index=True)

img_paths = []
titles = []
for r in picked.itertuples():
    p = first_image_for_run(r.run_dir, split='test')
    if p is not None:
        img_paths.append(p)
        titles.append(f"{r.ablation_label} | macro={r.test_macro_dice:.3f}")

if not img_paths:
    print('No qualitative images found in selected runs.')
else:
    cols = 3
    rows = int(np.ceil(len(img_paths) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.2 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, p, t in zip(axes, img_paths, titles):
        img = plt.imread(p)
        ax.imshow(img)
        ax.axis('off')

    for ax in axes[len(img_paths):]:
        ax.axis('off')

    plt.tight_layout()
    out = FIG_DIR / 'fig6_qualitative_top_bottom.png'
    fig.savefig(out, transparent=True)
    print('Wrote', out)
    plt.show()

## Exported Artifacts

This notebook writes:
- `ablation_main_table.csv`
- `ablation_onehot_gain_vs_baseline.csv`
- `figures/*.png`

All outputs are stored under:
`/scratch/work/sethih1/Multimodal_TERS/paper_results/ablation_no_optuna`